In [1]:
!pip install -qU bitsandbytes peft huggingface_hub fsspec datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.3.1 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
jupyter-ai 2.31.5 requires faiss-cpu!=1.8.0.post0,<2.0.0,>=1.8.0, which is not installed.
autogluon-multimodal 1.3.1 requires nltk<3.9,>=3.4.5, but you have nltk 3.9.1 which is incompatible.
autogluon-multimodal 1.3.1 requires transformers[sentencepiece]<4.50,>=4.38.0, but you have transformers 4.53.1 which is incompatible.
autogluon-timeseries 1.3.1 requires transformers[sentencepiece]<4.50,>=4.38.0, but you have transformers 4.53.1 which is incompatible.
jupyter-scheduler 2.11.0 requires fsspec!=2025.3.1,<=2025.3.2,>=2023.6.0, but you have fsspec 2025.9.0 which is incompatible.
s3fs 2024.12.0 requires fsspec==2024.12.0.*, but you have fsspec 2025.9.0 which is incompatible.


In [19]:
!pip install wandb -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [1]:
import wandb

wandb.login(relogin=True)

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/sagemaker-user/.netrc
wandb: Currently logged in as: christophe-reigner (christophe-reigner-axa-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import os
from transformers import BitsAndBytesConfig
import subprocess as sp
import os
from datetime import datetime

2025-10-28 19:54:03.980391: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761681243.996153   22369 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761681244.001089   22369 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-28 19:54:04.016732: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def optimize_model_memory(model):
    """
    Optimizes the model to use less memory during training.

    Args:
        model: The language model to optimize.

    Returns:
        The optimized model.

    Explanation:
        1. Sets the model to training mode.
        2. Disables KV caching to save memory.
        3. Enables gradient checkpointing to trade computation for memory.
        4. Ensures that input embeddings require gradients:
           - Either uses the built-in method if available.
           - Or adds a forward hook to the input embeddings layer.
        5. Returns the optimized model ready for memory-efficient training.
    """
    model.train()
    model.config.use_cache = False

    # First ensure inputs will require gradients
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    else:
        def make_inputs_require_grad(module, input, output):
            output.requires_grad_(True)
        model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

    # Then enable gradient checkpointing
    model.gradient_checkpointing_enable()

    return model

def get_gpu_memory():
    num_gpus = torch.cuda.device_count()
    command = "nvidia-smi --query-gpu=memory.free --format=csv"
    memory_free_info = sp.check_output(command.split()).decode('ascii').split('\n')[:-1][1:]
    memory_free_values = [int(x.split()[0]) for i, x in enumerate(memory_free_info)]
    print(f'{num_gpus} GPU(s) with free space: {memory_free_values}')

get_gpu_memory()

1 GPU(s) with free space: [22503]


In [3]:
# Free memory
#del model
#del trainer
with torch.no_grad():
    torch.cuda.empty_cache()
get_gpu_memory()

1 GPU(s) with free space: [22503]


In [4]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")
ds

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

## Test an off the sheldf very small MoE

In [7]:
model_name = "allenai/OLMoE-1B-7B-0924"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically selects available GPU
    #dtype=torch.float16,
    quantization_config=quantization_config
)
model = optimize_model_memory(model)
get_gpu_memory()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

1 GPU(s) with free space: [8561]


In [11]:
model.eval()

OlmoeForCausalLM(
  (model): OlmoeModel(
    (embed_tokens): Embedding(50304, 2048, padding_idx=1)
    (layers): ModuleList(
      (0-15): 16 x OlmoeDecoderLayer(
        (self_attn): OlmoeSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): OlmoeRMSNorm((2048,), eps=1e-05)
          (k_norm): OlmoeRMSNorm((2048,), eps=1e-05)
        )
        (mlp): OlmoeSparseMoeBlock(
          (gate): Linear(in_features=2048, out_features=64, bias=False)
          (experts): ModuleList(
            (0-63): 64 x OlmoeMLP(
              (gate_proj): Linear(in_features=2048, out_features=1024, bias=False)
              (up_proj): Linear(in_features=2048, out_features=1024, bias=False)
              (down_proj): Linear(in

In [13]:
def log_expert_activations(model_outputs):
    if hasattr(model_outputs, "router_logits"):
        router_logits = model_outputs.router_logits
        if isinstance(router_logits, tuple):
            router_logits = router_logits[0]
        expert_ids = torch.argmax(router_logits, dim=-1)
        print("Activated experts:", expert_ids.tolist())

In [14]:
for example in ds['train'].select(range(5)):
    input_text = example["question"]
    inputs = tokenizer(input_text, return_tensors="pt").to('cuda')
    with torch.no_grad():
        outputs = model(**inputs, output_router_logits=True)
        log_expert_activations(outputs)
        generated_ids = model.generate(**inputs, max_length=128)
        answer = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        print("Q:", input_text)
        print("A:", answer)

Activated experts: [26, 26, 50, 17, 59, 11, 35, 29, 6, 1, 10, 26, 45, 36, 2, 2, 1, 59, 25, 62, 55, 10, 26, 6, 6, 14, 55, 10, 58, 44, 17, 59, 16, 26, 45, 2, 45]
Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
A: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

A. 48
B. 72
C. 96
D. 120

Correct Answer: C. 96

Explanation:

Natalia sold 48 clips in April and then sold half as many clips in May.

So, in April, Natalia sold 48 clips.

In May, she sold half of 48, which is 24.

So, in May, Natalia sold 24 clips.
Activated experts: [62, 62, 17, 33, 10, 61, 29, 4, 49, 33, 44, 10, 6, 15, 20, 49, 36, 1, 47, 33, 29, 46, 18, 44, 10, 59, 15, 14, 55, 48, 1, 6]
Q: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
A: Weng e


KeyboardInterrupt



### OlMoE 1 GPU Full load

Using allenai/OLMoE-1B-7B-0924, if ran in full, OOM on cuda with 22GBVRAM straightaway. There is no great way to run a full supervised fine tuning.

### OlMoE 1 GPU - 8bit quantized

Works fine on 1 GPU
Training for 6 hours 

In [10]:
def preprocess(example):
    # Format: question as prompt, answer as target
    prompt = example['question'] + '\nAnswer:'
    target = example['answer']
    # Concatenate prompt and answer for causal LM
    full_text = prompt + ' ' + target
    tokenized = tokenizer(full_text, truncation=True, max_length=256, padding='max_length')
    tokenized['labels'] = tokenized['input_ids']
    return tokenized

train_data = ds['train'].map(preprocess)
val_data = ds['test'].map(preprocess)

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [7]:
model_name = "allenai/OLMoE-1B-7B-0924"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically selects available GPU
    #dtype=torch.float16,
    quantization_config=quantization_config
)
model = optimize_model_memory(model)
get_gpu_memory()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

1 GPU(s) with free space: [8561]


In [10]:
from peft import LoraConfig

lora_config = LoraConfig(
    target_modules=["q_proj", "k_proj"],
    modules_to_save=["lm_head"],
)

model.add_adapter(lora_config, adapter_name="lora")

In [11]:
print(model.device)
print(get_gpu_memory())

cuda:0
1 GPU(s) with free space: [8557]
None


## Training

In [16]:
training_args = TrainingArguments(
    output_dir='./moe_gsm8k_results',
    per_device_train_batch_size=,
    num_train_epochs=1,
    logging_steps=100,
    save_steps=1000,
    report_to=['wandb'],  # Enable Weights & Biases logging
    run_name="moe-test-1",
    #evaluation_strategy="steps",  # Evaluate every 'eval_steps'
    eval_steps=200,
    #fp16=torch.cuda.is_available(),
    #peft_config=Lora_config,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
)

In [17]:
trainer.train()

ValueError: expected sequence of length 228 at dim 1 (got 151)

## Test on Phi MoE

| Model                  | # Total param | # Act. param | MMLU  | MMLU pro | BBH   | Arc-C (chat) | Human-eval | GSM8K | MT-bench |
|------------------------|----------------|--------------|-------|----------|-------|--------------|------------|-------|----------|
| **MoE Models**         |                |              |       |          |       |              |            |       |          |
| Phi 3.5-MoE           | 42B            | 6.6B         | 78.36 | 59.38    | 63.93 | 91.38        | 81.70      | 87.87 | 8.34     |
| Qwen 1.5 MoE           | 14B            | 2.7B         | 60.73 | 26.49    | 42.65 | 67.24        | 46.30      | 53.07 | 6.55     |
| DeepSeek V2 Lite       | 16B            | 2.4B         | 56.69 | 17.89    | 36.30 | 61.09        | 54.40      | 63.23 | 6.82     |
| OL-MoE                | 7B             | 1.3B         | 54.27 | 20.87    | 38.00 | 55.63        | 37.80      | 71.49 | 6.60     |
| Granite 3.0 MoE        | 3.4B           | 0.8B         | 50.06 | 4.82     | 39.65 | 56.06        | 51.80      | 60.12 | 6.91     |
| **Dense Models**        |                |              |       |          |       |              |            |       |          |
| LLaMA 3.1 8B           | 8B             | 8B           | 68.71 | 45.28    | 50.86 | 82.42        | 69.50      | 84.84 | 8.03     |
| Qwen 2.5 7B            | 7.6B           | 7.6B         | 73.47 | 56.24    | 53.74 | 88.82        | 81.70      | 84.84 | 8.34     |
| Phi 3 small            | 7.4B           | 7.4B         | 75.35 | 52.06    | 62.07 | 84.30        | 70.10      | 84.84 | 8.03     |
| Gemma 3 4B             | 4B             | 4B           | 59.49 | 40.13    | 49.45 | 75.85        | 67.10      | 78.92 | 8.28     |
| Phi 3 mini             | 3.8B           | 3.8B         | 69.94 | 45.65    | 54.94 | 85.58        | 72.60      | 84.61 | 7.46     |
| LLaMA 3.2 3B           | 3.2B           | 3.2B         | 61.73 | 36.70    | 45.46 | 75.77        | 52.40      | 77.41 | 7.46     |
| Qwen 2.5 3B            | 3B             | 3B           | 65.06 | 41.00    | 46.61 | 80.20        | 73.80      | 76.57 | 7.60     |
| Gemma 3 1B             | 1B             | 1B           | 40.80 | 14.70    | 34.80 | 37.46        | 41.50      | 41.77 | 6.67     |
| LLaMA 3.2 1B           | 1B             | 1B           | 46.30 | 18.67    | 35.18 | 49.91        | 35.40      | 44.96 | 5.23     |
| **Our (SlimMoE) Models** |                |              |       |          |       |              |            |       |          |
| Phi-mini-MoE           | 7.6B           | 2.4B         | 70.68 | 49.68    | 55.27 | 84.91        | 73.80      | 84.89 | 7.59     |
| Phi-tiny-MoE           | 3.8B           | 1.1B         | 60.83 | 36.34    | 45.58 | 76.37        | 58.50      | 78.47 | 7.05     |


| Method                         | Description | Impact on Speed | Impact on Memory |
|--------------------------------|--------------|----------------|------------------|
| **Batch Size**                | Increase the batch size significantly to improve training speed, assuming sufficient GPU resources. | Yes, improves speed | Yes, has a significant impact on GPU memory |
| **Gradient Accumulation**       | Overcomes memory constraints by accumulating gradients over multiple mini-batches before updating parameters. Useful for fitting large models. Can slow down training due to extra forward/backward passes. | Yes, can reduce speed | Yes, reduces memory usage |
| **Gradient Checkpointing**       | Reduces memory by storing only some activations during the backward pass and recomputing others, avoiding storing all activations from the forward pass. | Yes, slower by ~20% | Yes, reduces memory usage |
| **Mixed Precision**              | Accelerates training by performing calculations in half-precision (fp16) where possible, maintaining some in full precision to preserve accuracy. | Yes | No, may increase memory usage due to dual fp16/fp32 models |
| **Optimizer Choice**             | Different optimizers can impact training performance and memory. | Yes | Yes |
| **Data Loader**                  | Use `pin_memory=True` for faster data transfer; set `num_workers=N` for preloading data with multiple processes. | Yes | No |
| **Free PyTorch Cache**           | Free or disable PyTorch cache to save memory (`torch.empty_cache()` or `model.config.use_cache=False`). | Slow down by ~10% | Yes, reduces memory |
| **Torch.compile**                | Compiles PyTorch code into optimized kernels for faster training using TorchDynamo. | Yes, improves speed | No |
| **Offload to CPU**                | Offloads optimizer state and model parameters to CPU when not in use, freeing GPU memory. | Yes | Yes, may slow down training |


In [ ]:
Flash attention family

### Basic evaluation

In [7]:
model_name = "microsoft/Phi-mini-MoE-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically selects available GPU
)
get_gpu_memory()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


1 GPU(s) with free space: [3251]


In [9]:
model.eval()

PhimoeForCausalLM(
  (model): PhimoeModel(
    (embed_tokens): Embedding(32064, 4096)
    (layers): ModuleList(
      (0-31): 32 x PhimoeDecoderLayer(
        (self_attn): PhimoeSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=True)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=True)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=True)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=True)
        )
        (block_sparse_moe): PhimoeSparseMoeBlock(
          (gate): Linear(in_features=4096, out_features=16, bias=False)
          (experts): ModuleList(
            (0-15): 16 x PhimoeBlockSparseTop2MLP(
              (w1): Linear(in_features=4096, out_features=960, bias=False)
              (w2): Linear(in_features=960, out_features=4096, bias=False)
              (w3): Linear(in_features=4096, out_features=960, bias=False)
              (act_fn): SiLU()
            )
          )
        )

In [16]:
import re

def extract_answer(text):
    # Extract numerical answer from model output
    match = re.search(r'####\s*(\d+)', text)
    if match:
        return int(match.group(1))
    # Fallback: look for last number in text
    numbers = re.findall(r'\d+', text)
    return int(numbers[-1]) if numbers else None

In [18]:
num_samples = 10

correct = 0

for i, example in enumerate(ds['test'].select(range(num_samples))):
    question = example["question"]
    answer = int(example["answer"].split("####")[-1].strip())
    
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predicted = extract_answer(response)
    
    if predicted == answer:
        correct += 1
        
    if i % 10 == 0:
        print(f"Progress: {i}/{num_samples}, Accuracy: {correct/(i+1):.3f}")

Progress: 0/10, Accuracy: 0.000



KeyboardInterrupt



In [ ]:
Add here inference evaluation ??

## Training SFT

In [ ]:
# Without quantization

# RuntimeError: You can't move a model that has some modules offloaded to cpu or disk. --> not enough GPU memory
model_name = "microsoft/Phi-mini-MoE-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically selects available GPU
    torch_dtype=torch.float16, # torch.bfloat16
)
#model = optimize_model_memory(model)
get_gpu_memory()

In [5]:
# With quantization - NOT WORKING !

# RuntimeError: You can't move a model that has some modules offloaded to cpu or disk. --> not enough GPU memory
model_name = "microsoft/Phi-mini-MoE-instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    #bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically selects available GPU
    quantization_config=quantization_config
)
model = optimize_model_memory(model)
get_gpu_memory()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

1 GPU(s) with free space: [14447]


In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    target_modules="all-linear",
    inference_mode=False, # set to False for training
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

model.add_adapter(lora_config, adapter_name="lora")

In [7]:
def preprocess(example):
    # Format: question as prompt, answer as target
    prompt = example['question'] + '\nAnswer:'
    target = example['answer']
    # Concatenate prompt and answer for causal LM
    full_text = prompt + ' ' + target
    tokenized = tokenizer(full_text, truncation=True, max_length=256, padding='max_length')
    tokenized['labels'] = tokenized['input_ids']
    return tokenized

train_data = ds['train'].map(preprocess)
val_data = ds['test'].map(preprocess)

In [9]:
os.environ["WANDB_PROJECT"] = "MoE"
training_args = TrainingArguments(
    output_dir='./moe_gsm8k_results',
    per_device_train_batch_size=32,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=100,
    report_to=['wandb'],  # Enable Weights & Biases logging
    run_name=f"moe-phi-{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    eval_strategy="steps",  # Evaluate every 'eval_steps'
    eval_steps=50,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    # minimal impact
    #bf16=True, # no impact when already using quantization
    #optim="adamw_bnb_8bit",
    dataloader_pin_memory=True,
    dataloader_num_workers=4,
    torch_compile=True,
    torch_compile_backend="inductor"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
)

In [ ]:
trainer.train()

wandb: Currently logged in as: christophe-reigner (christophe-reigner-axa-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


In [ ]:
# only 15 steps in total - why ? takes half the time in speed (batch of 32 constant ?)

In [5]:
model_name = "microsoft/Phi-mini-MoE-instruct"

# 4h training using this quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=False)
    #bnb_4bit_quant_type="nf4",
    #bnb_4bit_compute_dtype=torch.bfloat16,
    #bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically selects available GPU
    #dtype=torch.bfloat16,
    quantization_config=quantization_config
)
#model = optimize_model_memory(model)
get_gpu_memory()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

1 GPU(s) with free space: [14213]


In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    #target_modules=["q_proj", "k_proj"],
    target_modules=["q_proj", "v_proj"],
    #modules_to_save=["lm_head"],
    #lora_dropout=0.5,
)

model = get_peft_model(model, lora_config)

In [12]:
training_args = TrainingArguments(
    output_dir='./moe_gsm8k_results',
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=100,
    save_steps=500,
    report_to=['wandb'],  # Enable Weights & Biases logging
    run_name="moe-test-1",
    #evaluation_strategy="steps",  # Evaluate every 'eval_steps'
    eval_steps=200,
    #fp16=torch.cuda.is_available(),
    #peft_config=Lora_config,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
)

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [13]:
print(model.device)
print(get_gpu_memory())

cuda:0
1 GPU(s) with free space: [14491]
None


In [14]:
trainer.train()

Step,Training Loss
100,0.539800
200,0.407500
300,0.382400
400,0.362000
500,0.342800
600,0.352100
700,0.329800
800,0.347700
900,0.352600
1000,0.325600


TrainOutput(global_step=7473, training_loss=0.3307315727386005, metrics={'train_runtime': 9002.5732, 'train_samples_per_second': 0.83, 'train_steps_per_second': 0.83, 'total_flos': 6.306732005854464e+16, 'train_loss': 0.3307315727386005, 'epoch': 1.0})

In [16]:
print(get_gpu_memory())

1 GPU(s) with free space: [12947]
None


In [21]:
model.save_pretrained('phi_moe_281025')

# not working

trainer.evaluate(val_data)

# train v2 with better params

In [5]:
from peft import LoraConfig

In [6]:
model_name = "allenai/OLMoE-1B-7B-0924"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_v2 = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically selects available GPU
    quantization_config=quantization_config
)
model_v2 = optimize_model_memory(model_v2)

lora_config = LoraConfig(
    target_modules=["q_proj", "k_proj"],
    modules_to_save=["lm_head"],
)
model_v2.add_adapter(lora_config, adapter_name="lora")
get_gpu_memory()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

1 GPU(s) with free space: [15165]


In [7]:
def preprocess(example):
    # Format: question as prompt, answer as target
    prompt = example['question'] + '\nAnswer:'
    target = example['answer']
    # Concatenate prompt and answer for causal LM
    full_text = prompt + ' ' + target
    tokenized = tokenizer(full_text, truncation=True, max_length=256, padding='max_length')
    tokenized['labels'] = tokenized['input_ids']
    return tokenized

train_data = ds['train'].map(preprocess)
val_data = ds['test'].map(preprocess)

In [10]:
os.environ["WANDB_PROJECT"] = "MoE"
training_args = TrainingArguments(
    output_dir='./moe_gsm8k_results',
    per_device_train_batch_size=32,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=100,
    report_to=['wandb'],  # Enable Weights & Biases logging
    run_name=f"moe-phi-{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    eval_strategy="steps",  # Evaluate every 'eval_steps'
    eval_steps=50,
    #gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    # minimal impact
    #bf16=True, # no impact when already using quantization
    #optim="adamw_bnb_8bit",
    #dataloader_pin_memory=True,
    #dataloader_num_workers=4,
    #torch_compile=True,
    #torch_compile_backend="inductor"
)

trainer = Trainer(
    model=model_v2,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
)

In [11]:
trainer.train()

/opt/conda/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/opt/conda/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/opt/conda/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/opt/conda/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during

Step,Training Loss
100,4.118300
200,0.000000


TrainOutput(global_step=234, training_loss=1.7599407750317173, metrics={'train_runtime': 1532.1499, 'train_samples_per_second': 4.877, 'train_steps_per_second': 0.153, 'total_flos': 7.94338292096041e+16, 'train_loss': 1.7599407750317173, 'epoch': 1.0})

In [12]:
trainer.evaluate(val_data)

/opt/conda/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/opt/conda/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


{'eval_loss': nan,
 'eval_runtime': 246.0655,
 'eval_samples_per_second': 5.36,
 'eval_steps_per_second': 0.671,
 'epoch': 1.0}

## Optimize through quantization

The second level of optimization is to use quantization techniques.
This way we significantly reduce the memory consumption. We've used this technique above directly to avoid full SFT which consumed too much memory.

model_name = "microsoft/Phi-mini-MoE-instruct"

# 4h training using this quantization
#quantization_config = BitsAndBytesConfig(
#    load_in_4bit=True,
#    bnb_4bit_quant_type="nf4",
#    bnb_4bit_compute_dtype=torch.bfloat16,
#    bnb_4bit_use_double_quant=True)

# 4h training using this quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=False)
    #bnb_4bit_quant_type="nf4",
    #bnb_4bit_compute_dtype=torch.bfloat16,
    #bnb_4bit_use_double_quant=True)
# [14213] GPU remaining left ;  [52/7473 01:41 < 4:10:35] 


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically selects available GPU
    #dtype=torch.bfloat16,
    quantization_config=quantization_config
)
#model = optimize_model_memory(model)
get_gpu_memory()

# Why is quantization much longer ?

- conversion overhead: each forward/backward pass model:
   - de-quantize n-bit into FB16/F16 for numerical calculation
   - re-quantize for storage
   - x% increase by operation
- double quantization:
- lora operations:
- gradient checkpointing ; how much ?
- lora not all layers adapted ; how much ?

In [ ]:
Eval post training and eval from trainer not fixed at all

Add callback expert logger 

In [ ]:
gradient accumulation link 
https://lightning.ai/pages/blog/gradient-accumulation/